# MMJL Quickstart Demo

This notebook shows the normal, low-drama path for using Multimodal Jupy
Logger in a project notebook.

It demonstrates:

1. repository-root import setup
2. magic registration
3. literal Markdown logging
4. executed-cell capture
5. rich-display image capture
6. explicit file capture with `%jupy_file`
7. manifest inspection and timeline export


In [ ]:
## 1. Locate the repository root and configure imports.

import importlib
import os
import sys
from pathlib import Path

starting_dir = Path.cwd().resolve()
repo_root = None

for candidate_path in [starting_dir, *starting_dir.parents]:
    package_path = (
        candidate_path
        / "src"
        / "multimodal_jupy_logger"
    )

    if package_path.is_dir():
        repo_root = candidate_path
        break
    ##endof:  if package_path.is_dir()
##endof:  for candidate_path in [...]

if repo_root is None:
    raise RuntimeError(
        "Could not find repository root containing "
        "src/multimodal_jupy_logger."
    )
##endof:  if repo_root is None

src_path = repo_root / "src"
os.chdir(repo_root)

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
##endof:  if str(src_path) not in sys.path

importlib.invalidate_caches()

print("repo_root:", repo_root)
print("current working directory:", Path.cwd())


In [ ]:
## 2. Import MMJL and register the notebook magics.

from multimodal_jupy_logger import (
    MultimodalJupyLogger,
    register_jupy_logger,
)

logger = MultimodalJupyLogger(root=repo_root / "jupy_log")
register_jupy_logger()

print("manifest:", logger.manifest)
print("artifacts:", logger.artifacts)
print("staging:", logger.staging)
print("timelines:", logger.timelines)


## 3. Log a literal Markdown note

The next cell uses `%%jupy_log`. It stores the cell text without executing it.


In [ ]:
%%jupy_log --label quickstart-note --mime text/markdown
# Logged Markdown note

This Markdown was recorded by MMJL as a literal artifact.


## 4. Capture executed code

The next cell uses `%%jupy_capture`. It records the input code, stdout, and
rich-display plot output.


In [ ]:
%%jupy_capture --label quickstart-plot
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(20260618)
x_values = np.arange(1, 11)
y_values = x_values ** 2 + rng.integers(-5, 6, size=len(x_values))

print("x_values:", x_values)
print("y_values:", y_values)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x_values, y_values, marker="o")
ax.set_title("MMJL quickstart captured plot")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.grid(True)

plot_path = logger.staging / "quickstart_plot.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
print("saved explicit plot file:", plot_path)
print("explicit plot exists:", plot_path.exists())

plt.show()


## 5. Log an existing file

The plot above was already captured as rich notebook output. The next cell
also imports the saved PNG from MMJL's staging directory with `%jupy_file`.


In [ ]:
get_ipython().run_line_magic(
    "jupy_file",
    (
        "--label quickstart-explicit-plot "
        "--mime image/png "
        f'"{plot_path}"'
    ),
)


## 6. Inspect, validate, and export timelines


In [ ]:
%jupy_inspect


In [ ]:
%jupy_validate


In [ ]:
%jupy_markdown
%jupy_html


## 7. Done

Expected result: the manifest validates with zero missing artifacts, and
Markdown/HTML timelines are written under `jupy_log/timelines/`.
